# 02 · Import prepared data into SQL staging

**Fresh database builds only.** First run SQL 01 and 02 in SSMS. Input: core_analysis.pkl. Output: five populated staging tables. Existing nonempty targets deliberately stop the import; do not clear your working database to rerun this notebook. After import, run SQL 03–10 in the documented order.

Run cells from top to bottom in a **fresh kernel**. The setup cell declares inputs explicitly. Code comments and explanations are in English. The original source cell numbers are recorded in notebook metadata and in `docs/cell_mapping.csv`.

## Setup

Locate the repository and import shared utilities. This does not change the database.

In [ ]:
from pathlib import Path
import sys
import json
from urllib.request import urlopen
import numpy as np
import pandas as pd
from IPython.display import display

# Find the repository from any notebook subfolder.
root = next((p for p in [Path.cwd(), *Path.cwd().parents]
             if (p / "src" / "scouting" / "config.py").is_file()), None)
if root is None:
    raise FileNotFoundError("Open this notebook inside the extracted repository.")
if str(root / "src") not in sys.path:
    sys.path.insert(0, str(root / "src"))
from scouting.config import PROJECT_ROOT, RAW_DIR, PROCESSED_DIR, REPORTS_DIR, IMPECT_BASE_URL, ITERATION_ID
from scouting.data_access import load_or_download_json, load_metadata_context, metadata
from scouting.checkpoints import save_context, load_context, BASE_VARIABLES, ADVANCED_VARIABLES
print("Project:", PROJECT_ROOT)

globals().update(load_context("core_analysis"))
from scouting.database import require_load_enabled
RUN_DATABASE_LOAD = False  # Change to True only when importing into EMPTY staging tables.


## Check the SQL Server driver

Python needs an installed Microsoft ODBC driver as well as the pyodbc package.

In [ ]:
import pyodbc

print("Installed SQL Server drivers:")

sql_drivers = [
    driver
    for driver in pyodbc.drivers()
    if "SQL Server" in driver
]

for driver in sql_drivers:
    print(driver)

if not sql_drivers:
    print("No SQL Server ODBC driver found.")

## Connect using local configuration

Use Windows authentication and environment-configurable server/database names. This cell only checks connectivity.

In [ ]:
from scouting.database import connection_string as build_connection_string
connection_string = build_connection_string()
with pyodbc.connect(connection_string, timeout=10) as connection:
    cursor = connection.cursor()
    cursor.execute("SELECT DB_NAME() AS database_name;")
    print("Connected database:", cursor.fetchone()[0])


## Prepare players and teams

Rename provider fields to staging columns, parse birth dates and validate unique IDs before writing.

In [ ]:
players_to_load = players[
    ["id", "commonname", "firstname", "lastname", "birthdate", "leg"]
].rename(columns={
    "id": "player_id",
    "commonname": "player_name",
    "firstname": "first_name",
    "lastname": "last_name",
    "birthdate": "birth_date",
    "leg": "preferred_foot",
}).copy()

players_to_load["birth_date"] = pd.to_datetime(
    players_to_load["birth_date"],
    errors="raise",
).dt.date

teams_to_load = squads[
    ["id", "name", "countryId", "type", "gender"]
].rename(columns={
    "id": "team_id",
    "name": "team_name",
    "countryId": "country_id",
    "type": "team_type",
    "gender": "gender",
}).copy()

# Check identifiers before importing.
for table, id_column in [
    (players_to_load, "player_id"),
    (teams_to_load, "team_id"),
]:
    if table[id_column].isna().any():
        raise ValueError(f"Missing IDs in {id_column}")

    if table[id_column].duplicated().any():
        raise ValueError(f"Duplicate IDs in {id_column}")

print("Players ready:", len(players_to_load))
print("Teams ready:", len(teams_to_load))

display(players_to_load.head(3))
display(teams_to_load.head(3))

## Load players and teams transactionally

Refuse nonempty staging tables. Commit only if imported row counts match the prepared inputs.

In [ ]:
require_load_enabled(RUN_DATABASE_LOAD)

def sql_rows(table):
    # Convert Pandas missing values to SQL-compatible None.
    clean_table = table.astype(object).where(pd.notna(table), None)
    return list(clean_table.itertuples(index=False, name=None))


connection = pyodbc.connect(
    connection_string,
    timeout=10,
    autocommit=False,
)

try:
    cursor = connection.cursor()

    # Prevent accidentally appending the same data twice.
    cursor.execute("SELECT COUNT_BIG(*) FROM staging.players;")
    existing_players = cursor.fetchone()[0]

    cursor.execute("SELECT COUNT_BIG(*) FROM staging.teams;")
    existing_teams = cursor.fetchone()[0]

    if existing_players != 0 or existing_teams != 0:
        raise RuntimeError(
            "The staging tables already contain data. "
            "Import stopped to prevent duplicate rows."
        )

    cursor.executemany(
        """
        INSERT INTO staging.players (
            player_id,
            player_name,
            first_name,
            last_name,
            birth_date,
            preferred_foot
        )
        VALUES (?, ?, ?, ?, ?, ?);
        """,
        sql_rows(players_to_load),
    )

    cursor.executemany(
        """
        INSERT INTO staging.teams (
            team_id,
            team_name,
            country_id,
            team_type,
            gender
        )
        VALUES (?, ?, ?, ?, ?);
        """,
        sql_rows(teams_to_load),
    )

    # Verify both loads before committing.
    cursor.execute("SELECT COUNT_BIG(*) FROM staging.players;")
    loaded_players = cursor.fetchone()[0]

    cursor.execute("SELECT COUNT_BIG(*) FROM staging.teams;")
    loaded_teams = cursor.fetchone()[0]

    if (
        loaded_players != len(players_to_load)
        or loaded_teams != len(teams_to_load)
    ):
        raise RuntimeError("Imported row counts do not match Python.")

    connection.commit()

    print("Import committed successfully.")
    print("SQL player rows:", loaded_players)
    print("SQL team rows:", loaded_teams)

except Exception:
    connection.rollback()
    raise

finally:
    connection.close()

## Prepare matches

Convert kickoff timestamps to UTC DATETIME2 values and convert provider matchdays 0–33 to readable values 1–34.

In [ ]:
matches_to_load = impect_matches[
    [
        "id",
        "iterationId",
        "homeSquadId",
        "awaySquadId",
        "scheduledDate",
        "matchDay.index",
        "available",
    ]
].rename(columns={
    "id": "match_id",
    "iterationId": "iteration_id",
    "homeSquadId": "home_team_id",
    "awaySquadId": "away_team_id",
    "scheduledDate": "kickoff_utc",
    "matchDay.index": "matchday",
}).copy()

# Parse dates as UTC, then remove timezone metadata for DATETIME2.
# The stored clock values remain UTC.
matches_to_load["kickoff_utc"] = pd.to_datetime(
    matches_to_load["kickoff_utc"],
    utc=True,
    errors="raise",
).dt.tz_localize(None)

# IMPECT numbers matchdays 0–33; store readable values 1–34.
matches_to_load["matchday"] = matches_to_load["matchday"] + 1

if matches_to_load.isna().any().any():
    raise ValueError("Missing values found in the match import.")

if matches_to_load["match_id"].duplicated().any():
    raise ValueError("Duplicate match IDs found.")

known_team_ids = set(teams_to_load["team_id"])

for column in ["home_team_id", "away_team_id"]:
    if not matches_to_load[column].isin(known_team_ids).all():
        raise ValueError(f"Unknown team IDs in {column}")

print("Matches ready:", len(matches_to_load))
display(matches_to_load.head(3))

## Load match records

Use parameterized inserts, a nonempty-table guard and row-count verification before commit.

In [ ]:
require_load_enabled(RUN_DATABASE_LOAD)

match_rows = [
    (
        int(row.match_id),
        int(row.iteration_id),
        int(row.home_team_id),
        int(row.away_team_id),
        row.kickoff_utc.to_pydatetime(),
        int(row.matchday),
        bool(row.available),
    )
    for row in matches_to_load.itertuples(index=False)
]

connection = pyodbc.connect(
    connection_string,
    timeout=10,
    autocommit=False,
)

try:
    cursor = connection.cursor()

    cursor.execute("SELECT COUNT_BIG(*) FROM staging.matches;")

    if cursor.fetchone()[0] != 0:
        raise RuntimeError(
            "staging.matches already contains data. "
            "Import stopped to prevent duplicates."
        )

    cursor.executemany(
        """
        INSERT INTO staging.matches (
            match_id,
            iteration_id,
            home_team_id,
            away_team_id,
            kickoff_utc,
            matchday,
            available
        )
        VALUES (?, ?, ?, ?, ?, ?, ?);
        """,
        match_rows,
    )

    cursor.execute("SELECT COUNT_BIG(*) FROM staging.matches;")
    loaded_matches = cursor.fetchone()[0]

    if loaded_matches != len(matches_to_load):
        raise RuntimeError("SQL and Python match counts differ.")

    connection.commit()
    print("Import committed. SQL match rows:", loaded_matches)

except Exception:
    connection.rollback()
    raise

finally:
    connection.close()

## Prepare duration rows

Validate grain and required values, and use Decimal for the SQL duration/match-share columns.

In [ ]:
from decimal import Decimal

durations_to_load = season_durations[
    [
        "match_id",
        "squadId",
        "player.id",
        "player.position",
        "playDuration",
        "matchShare",
    ]
].rename(columns={
    "squadId": "team_id",
    "player.id": "player_id",
    "player.position": "position_name",
    "playDuration": "play_duration_seconds",
    "matchShare": "match_share",
}).copy()

duration_keys = [
    "match_id", "team_id", "player_id", "position_name"
]

if durations_to_load.isna().any().any():
    raise ValueError("Missing values found in playing-time records.")

if durations_to_load.duplicated(duration_keys).any():
    raise ValueError("Duplicate playing-time records found.")

if durations_to_load["play_duration_seconds"].lt(0).any():
    raise ValueError("Negative playing durations found.")

duration_rows = [
    (
        int(row.match_id),
        int(row.team_id),
        int(row.player_id),
        str(row.position_name),
        Decimal(str(row.play_duration_seconds)),
        Decimal(str(row.match_share)),
    )
    for row in durations_to_load.itertuples(index=False)
]

print("Playing-time records ready:", len(duration_rows))

## Load playing time

Check row counts and total seconds before committing, preventing duration inflation from a faulty transformation.

In [ ]:
require_load_enabled(RUN_DATABASE_LOAD)

connection = pyodbc.connect(
    connection_string,
    timeout=10,
    autocommit=False,
)

try:
    cursor = connection.cursor()

    cursor.execute("""
        SELECT COUNT_BIG(*)
        FROM staging.player_match_position;
    """)

    if cursor.fetchone()[0] != 0:
        raise RuntimeError(
            "Playing-time table already contains data. "
            "Import stopped to prevent duplicates."
        )

    cursor.executemany(
        """
        INSERT INTO staging.player_match_position (
            match_id,
            team_id,
            player_id,
            position_name,
            play_duration_seconds,
            match_share
        )
        VALUES (?, ?, ?, ?, ?, ?);
        """,
        duration_rows,
    )

    cursor.execute("""
        SELECT
            COUNT_BIG(*),
            SUM(play_duration_seconds)
        FROM staging.player_match_position;
    """)

    loaded_rows, sql_total_seconds = cursor.fetchone()

    expected_seconds = sum(row[4] for row in duration_rows)

    if loaded_rows != len(duration_rows):
        raise RuntimeError("SQL and Python row counts differ.")

    # Allow only a small aggregate difference from decimal rounding.
    if abs(sql_total_seconds - expected_seconds) > Decimal("0.01"):
        raise RuntimeError("SQL and Python duration totals differ.")

    connection.commit()

    print("Import committed.")
    print("SQL playing-time rows:", loaded_rows)
    print("SQL total seconds:", sql_total_seconds)
    print("Duration total reconciles with Python: YES")

except Exception:
    connection.rollback()
    raise

finally:
    connection.close()

## Prepare event inserts

Use a fixed source-to-SQL mapping and preserve missing actor/coordinate values as NULL.

In [ ]:
import math

# Source column → SQL column, in insertion order.
event_column_map = {
    "match_id": "match_id",
    "id": "event_id",
    "index": "event_index",
    "periodId": "period_id",
    "gameTime.gameTimeInSec": "game_time_seconds",
    "squadId": "team_id",
    "player.id": "player_id",
    "player.position": "position_name",
    "actionType": "action_type",
    "action": "action_detail",
    "result": "result",
    "phase": "phase",
    "pressure": "pressure",
    "start.adjCoordinates.x": "start_x",
    "start.adjCoordinates.y": "start_y",
    "end.adjCoordinates.x": "end_x",
    "end.adjCoordinates.y": "end_y",
}

integer_columns = {
    "match_id", "event_id", "event_index",
    "period_id", "team_id", "player_id",
}

float_columns = {
    "game_time_seconds", "pressure",
    "start_x", "start_y", "end_x", "end_y",
}

sql_event_columns = list(event_column_map.values())

def convert_event_value(value, column):
    if pd.isna(value):
        return None

    if column in integer_columns:
        number = int(value)
        if number != value:
            raise ValueError(f"Non-integer value in {column}: {value}")
        return number

    if column in float_columns:
        number = float(value)
        if not math.isfinite(number):
            raise ValueError(f"Non-finite value in {column}")
        return number

    return str(value)

# Column names come only from our fixed mapping above.
insert_events_sql = (
    "INSERT INTO staging.events ("
    + ", ".join(sql_event_columns)
    + ") VALUES ("
    + ", ".join(["?"] * len(sql_event_columns))
    + ");"
)

print("Events ready:", len(season_events))

## Load events in batches

Insert the season in batches inside one transaction. Any failure rolls back this run; a nonempty target is never silently appended.

In [ ]:
require_load_enabled(RUN_DATABASE_LOAD)

BATCH_SIZE = 5000
expected_rows = len(season_events)

connection = pyodbc.connect(
    connection_string,
    timeout=10,
    autocommit=False,
)

try:
    cursor = connection.cursor()

    cursor.execute("SELECT COUNT_BIG(*) FROM staging.events;")

    if cursor.fetchone()[0] != 0:
        raise RuntimeError(
            "staging.events already contains data. "
            "Import stopped to prevent duplicates."
        )

    cursor.fast_executemany = True

    for start in range(0, expected_rows, BATCH_SIZE):
        batch = season_events.iloc[
            start:start + BATCH_SIZE
        ][list(event_column_map)]

        batch_rows = [
            tuple(
                convert_event_value(value, column)
                for value, column in zip(row, sql_event_columns)
            )
            for row in batch.itertuples(index=False, name=None)
        ]

        cursor.executemany(insert_events_sql, batch_rows)

        processed = min(start + BATCH_SIZE, expected_rows)

        if processed % 50000 == 0 or processed == expected_rows:
            print(
                f"Inserted {processed:,}/{expected_rows:,} "
                "(not committed yet)",
                flush=True,
            )

    cursor.execute("SELECT COUNT_BIG(*) FROM staging.events;")
    loaded_events = cursor.fetchone()[0]

    if loaded_events != expected_rows:
        raise RuntimeError("SQL and Python event counts differ.")

    connection.commit()

    print("\nImport committed successfully.")
    print("SQL event rows:", loaded_events)

except Exception:
    connection.rollback()
    print("Import failed; inserts from this run were rolled back.")
    raise

finally:
    connection.close()